# 04 -- Denoising Diffusion Models

**Goal:** generate images by learning to *undo noise*. Destroy images gradually with Gaussian noise
(a fixed process), train a network to predict the noise, then start from pure noise and denoise step by step.

**In this notebook:** the forward process (visualised before any training), the DDPM objective, the
DDPM and DDIM samplers, classifier-free guidance, memorisation on small data, and Hub upload.

*Papers:* Ho et al., *DDPM* (2020) · Song et al., *DDIM* (2020) · Nichol & Dhariwal (2021) · Ho & Salimans, *Classifier-Free Guidance* (2022).

## 1. Theory

### Forward process (fixed)
Add a little Gaussian noise at each of $T=1000$ steps, $q(x_t\mid x_{t-1})=\mathcal N(\sqrt{1-\beta_t}\,x_{t-1},\,\beta_t I)$.
Because Gaussians compose, any step can be reached in **one shot**:
$$
x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon,\qquad \bar\alpha_t=\prod_{s\le t}(1-\beta_s),\quad \varepsilon\sim\mathcal N(0,I).
$$
At $t=T$, $\bar\alpha_T\approx0$: $x_T$ is pure noise whatever $x_0$ was.

### Reverse process (learned)
If the steps are small, the reverse $q(x_{t-1}\mid x_t)$ is also approximately Gaussian. We learn it with a
U-Net $\varepsilon_\theta(x_t,t)$ that predicts the noise. Ho et al. showed the variational bound reduces to a
remarkably simple loss:
$$
\mathcal L_\text{simple}=\mathbb E_{t,x_0,\varepsilon}\big\|\varepsilon-\varepsilon_\theta(\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\varepsilon,\;t)\big\|^2 .
$$
Predicting noise is (up to scaling) predicting the **score** $\nabla_x\log p_t(x)$: the network learns which way is "more like data" at every noise level.

### Sampling
* **DDPM** (ancestral): $x_{t-1}=\frac{1}{\sqrt{\alpha_t}}\big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\varepsilon_\theta\big)+\sigma_t z$, all 1000 steps.
* **DDIM**: estimate $\hat x_0=(x_t-\sqrt{1-\bar\alpha_t}\varepsilon_\theta)/\sqrt{\bar\alpha_t}$, then jump straight to an earlier $t'$:
  $x_{t'}=\sqrt{\bar\alpha_{t'}}\,\hat x_0+\sqrt{1-\bar\alpha_{t'}}\,\varepsilon_\theta$. Deterministic, works with ~50 steps, **same trained model**.

### Noise schedule
`linear` ($\beta$ from $10^{-4}$ to $0.02$) is the original; `cosine` destroys information more evenly and works
better at low resolution -- compare the two in the forward-process plot below.

### Classifier-free guidance
Train one network both conditionally and unconditionally (drop the label to a "null" class with probability
`p_uncond`). At sampling, extrapolate away from the unconditional prediction:
$\tilde\varepsilon=\varepsilon_\text{uncond}+w(\varepsilon_\text{cond}-\varepsilon_\text{uncond})$. $w>1$: sharper, more on-class, less diverse.
This is the mechanism behind text-to-image prompt adherence.

## 0. Setup

Run from the repo root or from `notebooks/`. The package is imported from the repo,
so editing `genmodels/*.py` and re-running a cell picks up the change after a kernel restart.

On **RunPod**: `bash scripts/runpod/setup.sh` once, then open JupyterLab (port 8888).
On **Colab**: uncomment the first line.

In [ ]:
# !git clone -q https://github.com/yar78/vision-generative-models-survey.git && %cd vision-generative-models-survey && pip install -q -e .
import sys, pathlib
REPO = pathlib.Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import torch
import yaml
from genmodels import build_trainer
from genmodels.common.config import load_config
from genmodels.common.data import get_dataset
from genmodels.common.utils import get_device, load_images, nearest_neighbours, plot_history, show_grid
from genmodels.hub import push_run, resolve_repo_id
from genmodels.diffusion import DiffusionModel, make_beta_schedule
print("device:", get_device())

## 2. Choose an experiment

Every experiment is one YAML file in `configs/`, shared with the command-line script,
so a run started here and a run started on a RunPod GPU are the same experiment.

* `configs/diffusion/ddpm_mnist.yaml` -- the quick default.
* `configs/ray_dog/ddpm.yaml` -- your **Ray_dog** photos (`yaryar78/Ray_dog`, 258 images) at 64x64.
* `configs/diffusion/ddpm_cifar10.yaml` -- the classic unconditional CIFAR-10 setup (long run).

Edit `cfg` below to change anything (steps, batch size, model width ...).

In [ ]:
CONFIG = REPO / "configs/diffusion/ddpm_mnist.yaml"        # or: REPO / "configs/ray_dog/ddpm.yaml"
cfg = load_config(CONFIG)

# Override anything here, for example:
# cfg["train"]["steps"] = 5000

print(yaml.safe_dump(cfg, sort_keys=False))

### Look at the training data (with the training-time augmentation applied)

In [ ]:
train_set = get_dataset(dict(cfg["data"], num_workers=0))
print(f"{len(train_set)} training images")
x_real = torch.stack([train_set[i][0] for i in torch.randperm(len(train_set))[:64].tolist()])
show_grid(x_real, title="training data")

### The forward process, before any training
One image noised to increasing $t$, under both schedules. Notice how much earlier the linear schedule turns the image into noise.

In [ ]:
x0 = x_real[:1]
ts = torch.tensor([0, 50, 100, 200, 300, 500, 700, 999])
rows = []
for sched in ("linear", "cosine"):
    abar = torch.cumprod(1 - make_beta_schedule(sched, 1000), 0).float()
    eps = torch.randn(len(ts), *x0.shape[1:])
    rows.append(abar[ts].sqrt()[:, None, None, None] * x0 + (1 - abar[ts]).sqrt()[:, None, None, None] * eps)
show_grid(torch.cat(rows), nrow=len(ts), title=f"q(x_t | x_0) at t = {ts.tolist()} -- linear (top), cosine (bottom)", size=10)

## 3. Build the model

`build_trainer(cfg)` creates the U-Net noise predictor and noise schedule, the optimizers and the training loop.
Outputs go to `outputs/<name>/` (on RunPod: `/workspace/genmodels/outputs/<name>/`, on the network volume).

In [ ]:
trainer = build_trainer(cfg)
print("run dir:", trainer.run_dir)
print({k: f"{v/1e6:.2f}M params" for k, v in trainer.num_params().items()})

### The training objective, as implemented

In [ ]:
import inspect; print(inspect.getsource(DiffusionModel.loss))

## 4. Train

`trainer.fit()` runs `train.steps` steps. Every `sample_every` steps it saves a fixed-seed
sample grid to `samples/`; every `ckpt_every` steps a checkpoint. Re-running the cell
(or the script) **resumes** from the checkpoint.

MNIST config: ~20 min on a GPU. Ray_dog: ~2-3 h for 40k steps. The loss drops fast then flattens -- it is an average over all noise levels, so judge progress by the sample grids.

Already trained this config with the script (e.g. on RunPod)? Set `TRAIN = False`
and the checkpoint in the run directory is loaded instead.

In [ ]:
TRAIN = True

if TRAIN:
    trainer.fit()
elif trainer.ckpt_path.exists():
    trainer.load_checkpoint()
    print(f"loaded checkpoint at step {trainer.step}")
else:
    print("No checkpoint yet in", trainer.run_dir)

### TensorBoard
Loss curves and sample grids over time. On RunPod, TensorBoard is also served on port 6006 by `scripts/runpod/train.sh`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {trainer.run_dir}/tensorboard

In [ ]:
plot_history(trainer.history)

## 5. Results
### Samples (DDIM, EMA weights)

In [ ]:
show_grid(trainer.sample_fixed(64), title='DDIM samples')

## 6. Inside the sampler
### The denoising trajectory
DDIM's running estimate of the clean image $\hat x_0$ as $t$ goes from 999 to 0: coarse layout appears first, details last.

In [ ]:
model = trainer.net_for_sampling("model").eval()
y = torch.arange(4, device=trainer.device) % model.num_classes if model.num_classes else None
torch.manual_seed(0)
_, traj = model.ddim_sample(4, steps=20, y=y, guidance_scale=trainer.guidance, device=trainer.device, return_trajectory=True)
picks = traj[:: 2] + [traj[-1]]
show_grid(torch.stack(picks, 1).flatten(0, 1), nrow=len(picks), title="x0 prediction along the trajectory", size=12)

### Number of sampling steps
Same starting noise, DDIM with 5 / 10 / 25 / 50 steps, and full 1000-step DDPM (slow on CPU).

In [ ]:
RUN_DDPM = trainer.device.type == "cuda"
rows = []
for steps in (5, 10, 25, 50):
    torch.manual_seed(0)
    rows.append(model.ddim_sample(8, steps=steps, y=(torch.arange(8, device=trainer.device) % model.num_classes) if model.num_classes else None,
                                  guidance_scale=trainer.guidance, device=trainer.device))
if RUN_DDPM:
    torch.manual_seed(0)
    rows.append(model.ddpm_sample(8, y=(torch.arange(8, device=trainer.device) % model.num_classes) if model.num_classes else None,
                                  guidance_scale=trainer.guidance, device=trainer.device))
show_grid(torch.cat(rows), nrow=8, title="DDIM 5 / 10 / 25 / 50 steps" + (" / DDPM 1000" if RUN_DDPM else ""), size=9)

### Classifier-free guidance (conditional configs only)
Each row is one guidance scale $w$; columns are the classes.

In [ ]:
if model.num_classes:
    y = torch.arange(model.num_classes, device=trainer.device)
    rows = []
    for w in (0.0, 1.0, 2.0, 4.0):
        torch.manual_seed(0)
        rows.append(model.ddim_sample(len(y), steps=50, y=y, guidance_scale=w, device=trainer.device))
    show_grid(torch.cat(rows), nrow=len(y), title="guidance w = 0, 1, 2, 4 (top to bottom)", size=9)
else:
    print("Unconditional model -- set method.conditional: true to try guidance.")

### Memorisation check

With a small dataset (Ray_dog has 258 images) a generative model can simply copy training
images. For each sample (top row) we show its nearest training image in pixel space (bottom
row). Near-identical pairs mean memorisation; similar-but-different means the model generalises.

In [ ]:
eval_set = get_dataset(dict(cfg["data"], random_crop=False, hflip=False, num_workers=0))
train_imgs = load_images(eval_set, max_images=10_000)
samples = trainer.sample_fixed(8).cpu()
nn_imgs = nearest_neighbours(samples, train_imgs)[:, 0]
show_grid(torch.cat([samples, nn_imgs]), nrow=8, title="samples (top) vs nearest training image (bottom)", size=10)

## 7. Push to the Hugging Face Hub

`push_run` uploads the run directory as a model repo **`<your-username>/{cfg['name']}`**
(private by default):

| file | |
|---|---|
| `model.safetensors`, `config.json` | weights + constructor args (`DiffusionModel.from_pretrained` reads these) |
| `README.md` | generated model card: dataset, steps, hardware, git commit, metrics |
| `samples.png`, `metrics.json`, `train_config.yaml` | results and the exact config |
| `runs/` | TensorBoard logs (shown in the Hub's *Training metrics* tab) |

Login once with a **write** token from https://huggingface.co/settings/tokens
(`notebook_login()` below, `huggingface-cli login`, or the `HF_TOKEN` env var). To use another
repo name set `cfg["hub"]["repo_id"]`; to publish, `cfg["hub"]["private"] = False` before training,
or `push_run(..., private=False)`.

In [ ]:
PUSH_TO_HUB = False

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login, whoami
    try:
        print("logged in as", whoami()["name"])
    except Exception:
        notebook_login()
    url = push_run(trainer.run_dir)

### Reload it from the Hub
This is how anyone (including future you) uses the model.

In [ ]:
if PUSH_TO_HUB:
    from genmodels.diffusion import DiffusionModel
    model = DiffusionModel.from_pretrained(resolve_repo_id(cfg)).to(trainer.device).eval()
    show_grid(model.generate(16, steps=50), title="reloaded from the Hub")

## 8. Things to try

* `method.schedule: cosine` vs `linear` on the same config and budget.
* DDIM with `eta=1.0` (stochastic) vs `eta=0` at 25 steps.
* Ray_dog: train to 40k steps and run the memorisation check at 10k / 20k / 40k -- when does it start copying?
* Compare against `05_flow_matching` at the same U-Net and steps (both configs are matched).